In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [12]:
key = pd.read_csv('key_inflation.csv')

In [14]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [15]:
key['date'] = key['date'].astype(str)
n = 13
for i in range(57, 69):
    n -= 1
    key.loc[i, 'date'] = f'{n}.2020'

In [16]:
# переводим даты в формат дат
key["date"] = pd.to_datetime(key["date"], format="%m.%Y")

In [17]:
key['key-rate'] = key['key-rate'].replace(',', '.', regex=True).astype(np.float64)
key['inflation'] = key['inflation'].replace(',', '.', regex=True).astype(np.float64)

In [18]:
key.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   date       145 non-null    datetime64[ns]
 1   key-rate   145 non-null    float64       
 2   inflation  145 non-null    float64       
dtypes: datetime64[ns](1), float64(2)
memory usage: 3.5 KB


In [22]:
key['inflation_target'] = 4 # добавляем колонку цель по инфляции = 4

In [23]:
key['difference'] = key['inflation'] - key['inflation_target'] # добавляем колонку разница между инфляцией и целью по инфляции

In [24]:
key['inflation_diff'] = key['inflation'].diff() # добавляем колонку разница межды инфляцией соседних дат

In [25]:
# заполняем пропуск медианой
med_inflation_diff = key["inflation_diff"].median()
key["inflation_diff"] = key["inflation_diff"].fillna(med_inflation_diff)

In [28]:
dollar = pd.read_csv('currency_rate.csv')
dollar["date"] = pd.to_datetime(dollar["date"], format="%d.%m.%Y")
dollar['dollar_rate'] = dollar['dollar_rate'].replace(',', '.', regex=True).astype(np.float64)
dollar_month = (dollar.set_index('date').resample('MS')['dollar_rate'].first().reset_index())
key = key.merge(dollar_month, on='date', how='left')

In [29]:
# добавляем колонку изменение ключевой ставки
key['change_key_rate'] = np.sign(key['key-rate'].diff()).astype('Int64').shift(-1)
med_change_key_rate = key["change_key_rate"].median()
key["change_key_rate"] = key["change_key_rate"].fillna(med_change_key_rate)

In [30]:
key.head(20)

,date,key-rate,inflation,inflation_target,difference,inflation_diff,dollar_rate,change_key_rate
0,2025-09-01,17.0,7.98,4,3.98,0.00,80.4261,1
1,2025-08-01,18.0,8.14,4,4.14,0.16,80.3163,0
2,2025-07-01,18.0,8.79,4,4.79,0.65,78.5284,1
3,2025-06-01,20.0,9.40,4,5.40,0.61,79.1285,1
4,2025-05-01,21.0,9.88,4,5.88,0.48,81.4933,0
5,2025-04-01,21.0,10.23,4,6.23,0.35,85.4963,0
6,2025-03-01,21.0,10.34,4,6.34,0.11,88.2568,0
7,2025-02-01,21.0,10.06,4,6.06,-0.28,97.8107,0
8,2025-01-01,21.0,9.92,4,5.92,-0.14,102.2911,0
9,2024-12-01,21.0,9.52,4,5.52,-0.40,107.1758,0


In [31]:
# делим на матрицу признаков и целевую переменную
X = key[['key-rate','inflation', 'inflation_target', 'difference', 'inflation_diff', 'dollar_rate']]
y = key['change_key_rate']

In [32]:
X.head()

,key-rate,inflation,inflation_target,difference,inflation_diff,dollar_rate
0,17.0,7.98,4,3.98,0.00,80.4261
1,18.0,8.14,4,4.14,0.16,80.3163
2,18.0,8.79,4,4.79,0.65,78.5284
3,20.0,9.40,4,5.40,0.61,79.1285
4,21.0,9.88,4,5.88,0.48,81.4933


In [33]:
y.head()

,change_key_rate
0,1
1,0
2,1
3,1
4,0


In [68]:
from sklearn.linear_model import  LogisticRegression
from sklearn.preprocessing import  MinMaxScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
# Разобьем данные на обучающую и тестовую выборки
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

In [69]:
X_train.head()

,key-rate,inflation,inflation_target,difference,inflation_diff,dollar_rate
36,7.50,13.68,4,9.68,1.05,60.2386
112,11.00,7.30,4,3.30,-0.20,66.1718
133,8.00,7.55,4,3.55,-0.48,35.4438
100,9.25,4.10,4,0.10,-0.30,56.9518
101,9.75,4.10,4,0.10,0.00,55.9606


In [70]:
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
model = LogisticRegression()
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

In [71]:
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.6666666666666666
